# Desafío Lab 17 — Forecasting con SARIMAX
**Curso:** Minería de Datos (EIN132A25)

## Contexto
Una aerolínea quiere proyectar la **demanda mensual de pasajeros** para planificar flota y personal en los próximos 24 meses. Tu trabajo es construir un modelo SARIMAX, validarlo y entregar un pronóstico con intervalo de confianza.

## Reglas del desafío
1. Trabaja sobre **este notebook**, llenando las celdas marcadas con `# TODO`.
2. **No** uses `train_test_split` aleatorio — el split debe ser temporal.
3. Reserva los **últimos 24 meses** como test set.
4. Justifica cada decisión (parámetros, transformaciones) en una celda Markdown breve.
5. El modelo final debe **superar al baseline naïve** (predicción = último valor observado) en MAPE.

## Entregables
- Pronóstico de 24 meses con intervalo al 95 %.
- Tabla comparativa de al menos 3 modelos (naïve, SARIMA, SARIMAX).
- Diagnóstico de residuos del modelo elegido.
- Conclusión en ≤ 5 líneas: ¿qué exógenas ayudaron y por qué?

## Evaluación
| Criterio | Puntos |
|----------|--------|
| EDA + descomposición correcta | 1.0 |
| Estacionariedad justificada (ADF + diff) | 1.0 |
| Lectura correcta de ACF/PACF | 1.0 |
| SARIMA ajustado y evaluado | 1.0 |
| SARIMAX con exógenas | 1.5 |
| Walk‑forward validation | 1.0 |
| Diagnóstico de residuos (Ljung‑Box) | 0.5 |
| Conclusión y discusión | 1.0 |

## 0. Setup

Si `statsmodels` no está instalado, descomenta la primera línea.

In [ ]:
# !pip install statsmodels pmdarima --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.figsize'] = (11, 4)
np.random.seed(42)

## 1. Dataset

**Airline Passengers** — totales mensuales de pasajeros internacionales (en miles), 1949‑1960. Es el dataset clásico de Box‑Jenkins, con tendencia + estacionalidad anual fuerte.

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
df.columns = ['pasajeros']
df.index.freq = 'MS'   # frecuencia mensual
print(df.shape)
print(df.head())
df['pasajeros'].plot(title='Pasajeros internacionales (miles), 1949-1960');

## 2. Split temporal

Los **últimos 24 meses** son test. No se tocan hasta la evaluación final.

In [ ]:
train = df.iloc[:-24]
test  = df.iloc[-24:]
print(f'Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} obs)')
print(f'Test : {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} obs)')

def metricas(y_true, y_pred, nombre):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true.values - np.asarray(y_pred)) / y_true.values)) * 100
    print(f'{nombre:30s} | MAE={mae:6.2f} | RMSE={rmse:6.2f} | MAPE={mape:5.2f}%')
    return {'modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

resultados = []

---
## Ejercicio 1 — EDA y descomposición STL (1.0 pt)

1. Grafica la serie con su media móvil de 12 meses y su desviación móvil. ¿La varianza es constante? ¿Conviene aplicar `log()`?
2. Descompón la serie (`train`) con STL usando `period=12`. Comenta tendencia y estacionalidad.

In [ ]:
# TODO 1.1 — rolling mean y rolling std sobre train['pasajeros']


# TODO 1.2 — STL(train['pasajeros'], period=12).fit() y graficar componentes


**Respuesta breve:** *(¿varianza creciente? ¿modelo aditivo o multiplicativo? ¿necesidad de log?)*

---
## Ejercicio 2 — Estacionariedad (1.0 pt)

1. Aplica el test ADF a la serie original, a `diff(1)` y a `diff(1).diff(12)`.
2. Decide qué `d` y `D` usar en SARIMA y justifica con los p‑values.

> **Recordatorio:** H₀ del ADF = raíz unitaria (NO estacionaria). Buscamos `p < 0.05`.

In [ ]:
def adf_report(serie, nombre):
    stat, p, *_ = adfuller(serie.dropna())
    veredicto = 'ESTACIONARIA' if p < 0.05 else 'NO estacionaria'
    print(f'{nombre:30s} | ADF={stat:7.3f} | p={p:.4f} | {veredicto}')

# TODO 2 — llamar a adf_report sobre: original, diff(1), diff(1).diff(12)
#          (sobre la serie log-transformada si decidieron usar log)


**Decisión:** `d = ?`, `D = ?` (con justificación).

---
## Ejercicio 3 — ACF / PACF (1.0 pt)

1. Grafica ACF y PACF de la serie ya diferenciada (la que quedó estacionaria en el ejercicio 2).
2. Propón valores iniciales de `(p, q)` para la parte no estacional y `(P, Q)` para la estacional (m=12).
3. Explica qué barra del PACF te dio `p` y qué barra del ACF te dio `q`.

In [ ]:
# TODO 3 — fig, axes = plt.subplots(1, 2, ...); plot_acf(...); plot_pacf(...)
#          con lags >= 36 para ver claramente el rezago 12.


**Propuesta:** `SARIMA(p,d,q)(P,D,Q,12)` = `( , , )( , , , 12)`.

---
## Ejercicio 4 — Baseline naïve + SARIMA (1.0 pt)

1. Calcula un **baseline naïve estacional**: la predicción del mes `t` es el valor del mes `t-12` observado en train.
2. Ajusta el SARIMA propuesto en el ejercicio 3 y pronostica los 24 meses de test.
3. Reporta MAE / RMSE / MAPE de ambos en la lista `resultados`.

In [ ]:
# TODO 4.1 — baseline naïve estacional (lag=12)
# pred_naive = ...
# resultados.append(metricas(test['pasajeros'], pred_naive, 'Naïve estacional'))


# TODO 4.2 — SARIMAX(train['pasajeros'], order=(p,d,q), seasonal_order=(P,D,Q,12)).fit()
#            forecast(steps=24) y registrar métricas


---
## Ejercicio 5 — SARIMAX con exógenas (1.5 pt)

No tenemos covariables externas en este dataset, así que construirás **features derivadas del calendario** como exógenas. Esto es una práctica estándar (regresión con errores ARIMA).

1. Construye al menos dos de estas exógenas para train **y** para test:
   - Términos de **Fourier** de orden 2 para estacionalidad anual: `sin(2πkt/12)`, `cos(2πkt/12)` con `k=1,2`.
   - Variable de **tendencia** lineal `t`.
   - Dummy de **temporada alta** (Jun‑Ago, por ejemplo).
2. Ajusta `SARIMAX(...)` con `exog=...` (puedes simplificar la parte estacional del SARIMA si las exógenas ya capturan el ciclo).
3. Pronostica los 24 meses pasando `exog=exog_test`. **Recuerda:** SARIMAX necesita las exógenas también en el horizonte de predicción.
4. Registra las métricas y compara con SARIMA puro.

In [ ]:
def construir_exog(index, t_offset=0):
    """Devuelve un DataFrame de exógenas alineado con `index`."""
    t = np.arange(len(index)) + t_offset
    # TODO 5.1 — construir columnas: sin1, cos1, sin2, cos2, tendencia, dummy_alta
    exog = pd.DataFrame({
        # 'sin1': ...,
        # 'cos1': ...,
        # 'tendencia': ...,
    }, index=index)
    return exog

# exog_train = construir_exog(train.index, t_offset=0)
# exog_test  = construir_exog(test.index,  t_offset=len(train))

# TODO 5.2 — ajustar SARIMAX con exog y forecast con exog_test
# TODO 5.3 — registrar métricas y agregar a `resultados`


**Pregunta:** ¿Cuál de las exógenas resultó significativa? Revisa `model.summary()` y mira los p‑values.

---
## Ejercicio 6 — Walk‑forward validation (1.0 pt)

El `forecast(steps=24)` predice los 24 meses **de una vez** con información solo hasta el final de train. En operación normalmente reentrenamos cada mes con la observación recién llegada.

1. Implementa walk‑forward sobre los 24 meses de test: en cada paso reentrena con todo el histórico disponible y predice **un solo mes**.
2. Compara MAPE del walk‑forward vs el forecast directo de 24 pasos. ¿En cuánto mejora?

In [ ]:
# TODO 6 — loop sobre test.index, refit en cada paso, forecast(steps=1)
#          historia_y = train['pasajeros'].copy()
#          historia_x = exog_train.copy()
#          preds_wf = []
#          for fecha in test.index:
#              ...


---
## Ejercicio 7 — Diagnóstico de residuos + intervalo de confianza (0.5 pt)

1. Sobre el modelo ganador llama a `plot_diagnostics(figsize=(12,8))`.
2. Aplica `acorr_ljungbox` con `lags=[12, 24]`. Buscas `p > 0.05` (residuos sin autocorrelación).
3. Usa `get_forecast(steps=24, exog=...)` para obtener la predicción media e intervalo al 95 %. Grafica train, test, predicción y la banda sombreada.

In [ ]:
# TODO 7.1 — plot_diagnostics del modelo ganador


# TODO 7.2 — Ljung-Box
# lb = acorr_ljungbox(model.resid, lags=[12, 24], return_df=True)


# TODO 7.3 — get_forecast + conf_int + gráfico con banda


---
## Ejercicio 8 — Tabla final y conclusión (1.0 pt)

1. Convierte `resultados` en un DataFrame ordenado por MAPE.
2. Escribe una conclusión de **máximo 5 líneas**: qué modelo ganó, qué exógenas aportaron, qué supuestos quedaron débiles y qué probarías a continuación (Prophet, LSTM, exógenas reales como PIB o feriados).

In [ ]:
# TODO 8 — pd.DataFrame(resultados).sort_values('MAPE')


**Conclusión:**

*(escribe aquí)*

---
## Bonus opcional (+0.5 pt)

Usa `pmdarima.auto_arima` con `seasonal=True, m=12, stepwise=True` para que la librería busque automáticamente la mejor combinación de parámetros. Compara el modelo automático contra el tuyo. ¿Eligió los mismos `(p,d,q)(P,D,Q)`?

```python
# !pip install pmdarima --quiet
# from pmdarima import auto_arima
# auto = auto_arima(train['pasajeros'], seasonal=True, m=12, stepwise=True, trace=True)
# print(auto.summary())
```